# decria-sec: QLoRA fine-tune of Qwen3 on a free Kaggle T4

End-to-end, reproducible fine-tuning of an open-source LLM (`Qwen/Qwen3`) into a bilingual (EN/PT)
assistant for **authorized** penetration testing and defense. This notebook runs the exact pipeline
from the GitHub repo [felipe-nantes/finetunning](https://github.com/felipe-nantes/finetunning),
with the dependency versions pinned in its `uv.lock`:

1. build the public instruction dataset (license-checked sources, filters, exact + MinHash dedup, stratified sampling)
2. generate synthetic Q&A **locally** with Ollama, anchored on MITRE ATT&CK, CWE Top 25, OWASP WSTG and tool docs
3. QLoRA training (NF4 4-bit + LoRA r=16, loss on the answer only, checkpoint/resume)
4. honest before/after evaluation: CyberMetric-500 accuracy with 95% CI, validation loss, fixed qualitative and refusal prompts
5. merge + GGUF export (Q4_K_M / Q8_0) with an Ollama `Modelfile`
6. publish adapter, GGUF and dataset to the Hugging Face Hub with generated cards

No paid APIs anywhere.

**Why Kaggle?** The project is designed around a 2016 GTX 1060 3GB (see the repo README for the
constraints that card imposes). The free T4 here is the documented plan B: same scripts, same data,
more VRAM. The model card records the GPU that actually trained the adapter.

## Before you run

In *Session options* (right panel):

- **Accelerator:** GPU T4 x1
- **Internet:** On
- **Add-ons > Secrets:** add `HF_TOKEN` (a Hugging Face token with *write* access) and attach it to this notebook.
  Without it the notebook still runs; it just skips the upload and leaves everything in the notebook output.

Then use **Save Version > Save & Run All (Commit)** so the run continues in the background.

In [ ]:
# ---- Run configuration -------------------------------------------------------
REPO_URL = "https://github.com/felipe-nantes/finetunning.git"
BRANCH = "main"

# configs/0.6b.yaml -> Qwen3-0.6B, the model the project targets on the GTX 1060 3GB (fastest run)
# configs/1.7b.yaml -> Qwen3-1.7B, only possible with >= 6 GB of VRAM (longer run, better answers)
CONFIG = "configs/0.6b.yaml"

SYNTH_TARGET = 1000                           # synthetic Q&A pairs to generate
GEN_MODEL = "qwen2.5:3b-instruct-q4_K_M"      # local generator served by Ollama
PUSH_TO_HUB = True                            # needs the HF_TOKEN secret

In [ ]:
import json
import os
import shutil
import subprocess
import time
import urllib.request


def sh(cmd: str, check: bool = True) -> int:
    """Run a shell command, stream its output into the cell, and stop the notebook on failure."""
    print(f"$ {cmd}", flush=True)
    proc = subprocess.Popen("set -o pipefail; " + cmd, shell=True, executable="/bin/bash",
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, errors="replace")
    for chunk in iter(lambda: proc.stdout.read(256), ""):
        print(chunk, end="", flush=True)
    code = proc.wait()
    if check and code != 0:
        raise RuntimeError(f"command failed with exit code {code}: {cmd}")
    return code


sh("nvidia-smi")
GPU_NAME = subprocess.check_output(
    "nvidia-smi --query-gpu=name --format=csv,noheader", shell=True, text=True
).splitlines()[0].strip()
print("GPU:", GPU_NAME)

## 1. Code and environment

The repo is cloned and its locked environment (Python 3.11, PyTorch CUDA 12.6 build, Transformers,
PEFT, TRL, bitsandbytes) is recreated with `uv sync`, isolated from Kaggle's preinstalled packages.

In [ ]:
WORK = "/kaggle/working"
REPO = f"{WORK}/finetunning"
os.environ["UV_PROJECT_ENVIRONMENT"] = "/tmp/decria-venv"   # keep the venv out of the notebook output
os.environ["UV_LINK_MODE"] = "copy"

if not os.path.isdir(REPO):
    sh(f"git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO}")
os.chdir(REPO)
sh("git log -1 --format='commit %h  %s'")

sh("pip install -q uv")
sh("uv sync")
sh("uv run python -c \"import torch, transformers, peft, trl, bitsandbytes as bnb; "
   "print('torch', torch.__version__, '| transformers', transformers.__version__, "
   "'| peft', peft.__version__, '| trl', trl.__version__, '| bitsandbytes', bnb.__version__); "
   "print('CUDA device:', torch.cuda.get_device_name(0))\"")

with open(CONFIG, encoding="utf-8") as fh:
    RUN_DIR = next(line.split(":", 1)[1].strip() for line in fh if line.startswith("output_dir:"))
print("config:", CONFIG, "| run dir:", RUN_DIR)

In [ ]:
# Unit tests (CPU) and the GPU gate: NF4 round trip, throughput and peak VRAM on this card.
# Informative here, so a failure does not stop the run.
sh("uv run pytest", check=False)
sh("uv run python scripts/00_check_env.py", check=False)

In [ ]:
HF_READY = False
if PUSH_TO_HUB:
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
        sh("uv run python -c \"from huggingface_hub import HfApi; print('Hugging Face user:', HfApi().whoami()['name'])\"")
        HF_READY = True
    except Exception as exc:
        print(f"HF_TOKEN not usable ({type(exc).__name__}). The run continues; artifacts stay in the notebook output.")

## 2. Public dataset

Two Apache-2.0 instruction datasets, filtered (English only, answer length, no generic refusals),
deduplicated (exact + MinHash 0.8) and sampled by theme (recon, web, network, privesc, defense, CVE).

In [ ]:
sh("uv run python scripts/01_build_dataset.py")

## 3. Synthetic data, generated locally

Ollama serves a small open model on this GPU. Each answer must cite terms from the licensed source
chunk it was generated from (MITRE ATT&CK, CWE Top 25, OWASP WSTG, nmap docs), 70% EN / 30% PT.

In [ ]:
sh("(command -v zstd > /dev/null) || (apt-get install -y -qq zstd > /dev/null 2>&1) || true", check=False)
sh("curl -fsSL https://ollama.com/install.sh | sh")

ollama_log = open(f"{WORK}/ollama.log", "w")
ollama = subprocess.Popen(["ollama", "serve"], stdout=ollama_log, stderr=subprocess.STDOUT)
try:
    for _ in range(90):
        try:
            urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
            break
        except Exception:
            time.sleep(1)
    else:
        raise RuntimeError("Ollama did not start; see /kaggle/working/ollama.log")

    sh(f"ollama pull {GEN_MODEL}")
    sh("uv run python scripts/02_gen_synthetic.py --fetch-seeds")
    sh(f"uv run python scripts/02_gen_synthetic.py --target {SYNTH_TARGET} --model {GEN_MODEL}")
finally:
    # Free the VRAM before training.
    ollama.terminate()
    try:
        ollama.wait(timeout=60)
    except subprocess.TimeoutExpired:
        ollama.kill()
    ollama_log.close()

sh("wc -l data/processed/train.jsonl data/processed/val.jsonl")
sh("cat data/manifest.json")
sh("nvidia-smi --query-gpu=memory.used,memory.total --format=csv")

## 4. QLoRA training

NF4 4-bit base + LoRA (r=16) on all attention and MLP projections, loss computed on the answer
only, cosine schedule, gradient checkpointing. If the session restarts, rerunning this cell resumes
from the last complete checkpoint.

In [ ]:
t0 = time.time()
sh(f"uv run python scripts/03_train.py --config {CONFIG}")
TRAIN_HOURS = round((time.time() - t0) / 3600, 2)
print(f"training wall time: {TRAIN_HOURS} h on {GPU_NAME}")

## 5. Evaluation: base vs adapter

Same quantization and prompts for both models. CyberMetric-500 is used for evaluation only and is
never part of the training data. Expectation written before seeing the numbers (see README): a small
multiple-choice gain at best; the visible change should be answer focus, format, language and
consistent refusal of out-of-scope requests.

In [ ]:
sh(f"uv run python scripts/04_eval.py --config {CONFIG}")

with open("docs/eval_results.json", encoding="utf-8") as fh:
    results = json.load(fh)
print(f"{'model':<10}{'MCQ acc':>10}{'95% CI':>20}{'val loss':>12}")
for name, r in results.items():
    ci = f"[{r['ci95'][0]:.3f}, {r['ci95'][1]:.3f}]"
    print(f"{name:<10}{r['mcq_acc']:>10.3f}{ci:>20}{r['val_loss']:>12.4f}")

In [ ]:
# Side-by-side sample answers (qualitative + refusal/regression prompts).
with open("docs/samples.md", encoding="utf-8") as fh:
    print(fh.read()[:6000])

## 6. Merge and GGUF export

The adapter is merged into the base weights on CPU, converted to GGUF with llama.cpp and quantized
to Q4_K_M and Q8_0. The generated `Modelfile` reproduces the training chat template for Ollama.

In [ ]:
LLAMA = f"{WORK}/llama.cpp"
if not os.path.isdir(LLAMA):
    sh(f"git clone --depth 1 https://github.com/ggml-org/llama.cpp {LLAMA}")
sh("(command -v cmake > /dev/null) || pip install -q cmake")
sh(f"cmake -S {LLAMA} -B {LLAMA}/build -DCMAKE_BUILD_TYPE=Release -DLLAMA_CURL=OFF > /dev/null")
sh(f"cmake --build {LLAMA}/build -j 4 --target llama-quantize | tail -n 3")
sh(f"uv pip install -q {LLAMA}/gguf-py sentencepiece protobuf")
sh(f"uv run --no-sync python scripts/05_merge_export.py --config {CONFIG} --llama-cpp {LLAMA}")
sh(f"ls -lh {RUN_DIR}/gguf")

## 7. Publish to the Hugging Face Hub

Adapter, GGUF and dataset go to three repos under the token owner's account, each with a generated
card (base model, licenses, evaluation table, intended use). `NOTICE` credits every data source.

In [ ]:
if HF_READY:
    sh(f"uv run --no-sync python scripts/06_push_hub.py --config {CONFIG} "
       f"--hours {TRAIN_HOURS} --gpu '{GPU_NAME} (Kaggle)'")
else:
    print("Skipped: no HF_TOKEN. Download the artifacts from the notebook output instead.")

In [ ]:
# Keep the useful artifacts in the notebook output and drop the bulky intermediates.
ART = f"{WORK}/artifacts"
os.makedirs(ART, exist_ok=True)
for src in ["docs/eval_results.json", "docs/samples.md", "data/manifest.json",
            "data/processed/train.jsonl", "data/processed/val.jsonl", f"{RUN_DIR}/gguf/Modelfile"]:
    if os.path.isfile(src):
        shutil.copy2(src, ART)
if os.path.isdir(f"{RUN_DIR}/adapter"):
    shutil.copytree(f"{RUN_DIR}/adapter", f"{ART}/adapter", dirs_exist_ok=True)
if os.path.isdir(f"{RUN_DIR}/gguf"):
    for name in os.listdir(f"{RUN_DIR}/gguf"):
        if name.endswith(".gguf") and not name.endswith("-f16.gguf"):
            shutil.copy2(f"{RUN_DIR}/gguf/{name}", ART)

os.chdir(WORK)
for bulky in [LLAMA, REPO]:
    shutil.rmtree(bulky, ignore_errors=True)
sh(f"ls -lh {ART}")

## Notes

- Hardware: this run used the Kaggle GPU printed in the first cell, not the GTX 1060 3GB the project
  is built around. The Hugging Face model card states the GPU and the measured training time.
- Scope: the assistant explains methodology, tools, output interpretation and mitigations for
  authorized security work. It is trained and evaluated to refuse functional malware and
  weaponized exploits.
- Everything here is reproducible from the repo: pinned dependencies, fixed seeds, published data.